# Agent's View: Editing the abl1 2D Test Cases

A walkthrough of the five abl1 2D benchmark cases (`benchmarks/cases/abl1_2d.jsonl`), showing each editing method (`mutate`, `grow`, `swap`, `remove`) and exactly what the agent receives as context: the group listing, per-group detail, the `matches` check, and the canonical ring-position feedback returned on a ring-changing edit.

In [1]:
from rdkit import Chem

from chemistree import DesignSession
from chemistree.commands import run_command

# The abl1 crystal ligand, shared by every 2D case.
LIG = "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ccc1F"


def canon(smiles):
    """Canonical SMILES for comparing an edit against the gold."""
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None

The `describe` method returns the fragment-group listing the agent works from — each group by id, name, ported SMILES, and how it connects.

In [2]:
ds = DesignSession(LIG, three_d=False)
print(ds.describe())

# Group Summary
Call describe_group(id) for a group's atom positions, rings, and topology.

- [0] heteroaromatic `[3*]c1nc([H])c2c([H])c([4*])c(=O)n([7*])c2n1` (C7H2N3O) — scaffold, attached to [3] amine, [2] benzene, [7] methyl
- [1] benzene `[1*]c1c([8*])c([H])c([H])c([2*])c1[H]` (C6H3) — scaffold, attached to [4] methyl, [3] amine, [8] fluoro
- [2] benzene `[4*]c1c([5*])c([H])c([H])c([H])c1[6*]` (C6H3) — scaffold, attached to [0] heteroaromatic, [5] chloro, [6] chloro
- [3] amine `[2*]N([3*])[H]` (HN) — scaffold, attached to [1] benzene, [0] heteroaromatic
- [4] methyl `[1*]C([H])([H])[H]` (CH3) — leaf, attached to [1] benzene
- [5] chloro `[5*]Cl` (Cl) — leaf, attached to [2] benzene
- [6] chloro `[6*]Cl` (Cl) — leaf, attached to [2] benzene
- [7] methyl `[7*]C([H])([H])[H]` (CH3) — leaf, attached to [0] heteroaromatic
- [8] fluoro `[8*]F` (F) — leaf, attached to [1] benzene

**Properties:** MW 429, cLogP 5.5, TPSA 60, HBD 1, HBA 4, RotB 3, aromatic rings 4, Fsp3 0.10, formal charg

## 1. Substitution — `sub-halogen`

*Replace fluorine with chlorine.* The fluoro is a leaf group; `swap` it for a chloro. A non-ring swap returns no position note.

Gold: `Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ccc1Cl`

In [3]:
ds = DesignSession(LIG, three_d=False)
print(run_command(ds, "swap 8 [8*]Cl"))
gold = "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ccc1Cl"
print("FINAL:", ds.smiles())
print("correct:", canon(ds.smiles()) == canon(gold))

swapped group 8
FINAL: Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ccc1Cl
correct: True


## 2. Substitution — `sub-ring-to-pyridine`

*Make the aniline ring a pyridine with N at the carbon para to the methyl.* `describe_group` gives the atom positions; `mutate` a ring carbon to N. The result now names the resulting ring and each port's canonical position.

Gold: `Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ncc1F`

In [4]:
ds = DesignSession(LIG, three_d=False)
print(ds.describe_group(1))

## [1] benzene `[1*]c1c([8*])c([H])c([H])c([2*])c1[H]` (C6H3) — scaffold, attached to [4] methyl, [3] amine, [8] fluoro
Atom ids are `:k` (use as grow/mutate position_id); `[n*]` = port (connects groups). Nearby atoms are named as `[element:id]` tokens, with `·[n*]` marking one that bears a port.
Positions list each heavy atom's neighbours by bond distance. On a plain benzene ring the first three are named ortho, meta, para (= 1, 2, 3 bonds; greek alpha, beta, gamma); every other relation — a fused ring included — is a plain bond count (`2 bonds`, `3 bonds`, …). Grow at a listed hydrogen id; mutate a heavy-atom id.

**Atom Map:** `[c:0]1([1*])[c:1]([H:9])[c:2]([2*])[c:3]([H:10])[c:4]([H:11])[c:5]1[8*]`
**Rings:**
- Ring A (6-membered): atoms 0, 1, 2, 3, 4, 5
**Positions:**
- atom 0 (aromatic C, bears [1*] methyl) — ortho: [c:1], [c:5]·[8*] fluoro; meta: [c:2]·[2*] amine, [c:4]; para: [c:3]
- atom 1 (aromatic C, H 9) — ortho: [c:0]·[1*] methyl, [c:2]·[2*] amine; meta: [c:3], [c:5]·[8*] 

In [5]:
# atom 3 is the ring carbon para to the methyl (atom 0); mutate it to nitrogen
print(run_command(ds, "mutate 1 3 N"))
gold = "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ncc1F"
print("correct:", canon(ds.smiles()) == canon(gold))

mutated position 3 to N; pyridine (IUPAC numbering): [2*] at C2, [1*] at C4, [8*] at C5 (open: C3, C6)
correct: True


## 3. Substitution — `swap-to-oxazole`

*Replace the fluoro-phenyl with a 1,3-oxazole, attaching at the 2-position, methyl at 5, F at 4.* A multi-port `swap` reconnects the existing substituents; the result reports where each port landed by canonical numbering, so the requested 2 / 4 / 5 placement is checkable at a glance.

Gold: `Cc1oc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)nc1F`

In [6]:
ds = DesignSession(LIG, three_d=False)
# amine [2*] at C2, methyl [1*] at C5, fluoro [8*] at C4 of the oxazole
print(run_command(ds, "swap 1 [2*]c1oc([1*])c([8*])n1"))
gold = "Cc1oc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)nc1F"
print("correct:", canon(ds.smiles()) == canon(gold))

swapped group 1; oxazole (IUPAC numbering): [2*] at C2, [8*] at C4, [1*] at C5
correct: True


## 4. Growing — `grow-halogen`

*Add a Cl ortho to the fluorine.* `grow` at the hydrogen ortho to the fluoro (from `describe_group`, atom 4's hydrogen id 11). Growing a leaf onto a plain benzene leaves a carbocycle, so no ring-position note.

Gold: `Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc(Cl)c1F`

In [7]:
ds = DesignSession(LIG, three_d=False)
print(run_command(ds, "grow 1 11 [*]Cl"))
gold = "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc(Cl)c1F"
print("correct:", canon(ds.smiles()) == canon(gold))

grew [*]Cl at position 11
correct: True


## 5. Core hopping — `core-hop-quinazoline`

*Replace the pyrido-pyrimidinone core with a quinazoline, aniline at the 2-position and dichlorophenyl at the 6-position.* `remove` the N-methyl, then a multi-port `swap`. The result names the new ring and each port's canonical position — the exact feedback that catches a swapped substituent — and `matches` confirms the ring identity.

Gold: `Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)ccc3n2)ccc1F`

In [8]:
ds = DesignSession(LIG, three_d=False)
print(run_command(ds, "remove 7"))
# aniline [3*] at C2 (between the two ring N), dichlorophenyl [4*] at C6
print(run_command(ds, "swap 0 [3*]c1ncc2cc([4*])ccc2n1"))
gold = "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)ccc3n2)ccc1F"
print("correct:", canon(ds.smiles()) == canon(gold))

removed group 7; group 0 now has an open hydrogen at position 13 where the group was attached — grow there to replace it
swapped group 0; quinazoline (IUPAC numbering): [3*] at C2, [4*] at C6 (open: C4, C5, C7, C8)
correct: True


`matches` is the objective check on ring identity — a quinazoline is not a quinoxaline:

In [9]:
print(run_command(ds, "matches quinazoline"))
print(run_command(ds, "matches quinoxaline"))

# Substructure `quinazoline` (quinazoline)

Whole molecule: 1 match.
Contained in group(s): [0] heteroaromatic.
Substituent positions: quinazoline (IUPAC numbering): N at C2, C at C6 (open: C4, C5, C7, C8)
# Substructure `quinoxaline` (quinoxaline)

Whole molecule: no match.
